# Convert HTML output from the Web Data Research Assistant to Excel

This notebook takes HTML output collected with the **Web Data Research Assistant (WDRA)**, developed by Leslie Carr, and turns each HTML file into a separate Excel file. Rows that contain no actual text are removed along the way.

The original script was created in 2022 and especially used for tweets. As mentioned below, the *Web Data Research Assistant* may no longer be fully functional now.

**Note:** This notebook runs via Live Code, directly in your browser. There is no access to a folder on your computer. Nothing is saved automatically, and everything is gone when you close the tab before dowloading the final file.

## About the scraping tool used to collect the web data

The Web Data Research Assistant was built as a **Chrome browser extension** that saved the data of the web pages you browse (including social media sites) as HTML files containing a data table. This notebook can convert these HTML files to EXCEL.

> However, the last update of the tool dates from 2023. We cannot be certain that it still works today, so you may not be able to collect new data with it. This notebook is still useful for HTML files that were collected earlier, or can be adapted to similar use cases.

The following training resources for the Web Data Research Assistant are available but need updates:

* A [tutorial document](https://www.southampton.ac.uk/~lac/WebDataResearchAssistant/HowTo.pdf) that explains how to use this software to investigate simple research questions, covering the use of Excel, Gephi and Voyant Tools to analyse the Twitter data.
* A set of summary [slides](https://www.southampton.ac.uk/~lac/WebDataResearchAssistant/Slides.pdf) based on the information from the tutorial.

---

## Step 1: Import Required Libraries

The first run can take a moment while the packages are loaded into your browser. `pandas` reads tabular data, `lxml` helps it understand HTML, and `openpyxl` writes Excel files. We also need tools for packing files into a zip archive and for talking to your browser (file upload and download).

In [ ]:
import micropip
await micropip.install(["pandas", "lxml", "openpyxl"])

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)  # hides the harmless pandas/pyarrow notice

import io
import zipfile
import numpy as np
import pandas as pd
from IPython.display import display, HTML
from js import document, Blob, URL, Object
from pyodide.ffi import create_proxy, to_js

print("Setup complete!")

## Step 2: Upload Your WDRA Files

Run the cell below. A **Choose files** button appears. Select one or more HTML files collected with the Web Data Research Assistant (the file names usually look like `WebDataRA-2022-09-21T11_52_44.html`). Hold **Ctrl** (Windows) or **Cmd** (Mac) to select several files in the file dialog.

Choosing again **replaces** the previous selection, so select all the files you want to process in one go.

In [ ]:
uploaded_files = {}   # filename -> HTML text of the file

display(HTML('<div id="wdra-upload-box"><input type="file" id="wdra-upload" accept=".html,.htm" multiple> '
             '<span id="wdra-upload-status">No files selected yet.</span></div>'))

file_input = document.getElementById("wdra-upload")
upload_status = document.getElementById("wdra-upload-status")

if file_input is None:
    # Fallback: add the controls to the page if the output was not yet in the DOM
    box = document.createElement("div")
    file_input = document.createElement("input")
    file_input.type = "file"
    file_input.accept = ".html,.htm"
    file_input.multiple = True
    upload_status = document.createElement("span")
    upload_status.textContent = " No files selected yet."
    box.appendChild(file_input)
    box.appendChild(upload_status)
    document.body.appendChild(box)

async def _read_files(event):
    uploaded_files.clear()
    n = file_input.files.length
    if n == 0:
        upload_status.textContent = "No files selected yet."
        return
    for i in range(n):
        f = file_input.files.item(i)
        uploaded_files[f.name] = await f.text()
    upload_status.textContent = f"Loaded {len(uploaded_files)} file(s). Continue with Step 3."

file_input.addEventListener("change", create_proxy(_read_files))

## Step 3: Extract the Tables and Remove Empty Tweets

For each uploaded file, the first table is read into a pandas DataFrame. Rows where the `Text` column is empty (or contains only spaces) are removed. Files without a table or without a `Text` column are skipped, with a message.

In [ ]:
cleaned_tables = {}   # filename -> cleaned DataFrame

if not uploaded_files:
    print("No files loaded yet. Go back to Step 2 and upload your files first.")
else:
    for filename, html in uploaded_files.items():
        try:
            tables = pd.read_html(io.StringIO(html), flavor="lxml")
        except ValueError:
            print(f"{filename}: no table found, skipped.")
            continue
        except Exception as e:
            print(f"{filename}: could not be read ({e}), skipped.")
            continue

        table = tables[0]
        if "Text" not in table.columns:
            print(f"{filename}: no 'Text' column (found: {', '.join(map(str, table.columns))}), skipped.")
            continue

        before = len(table)
        has_text = table["Text"].astype("string").str.strip().fillna("") != ""
        table = table[has_text].reset_index(drop=True)
        cleaned_tables[filename] = table
        print(f"{filename}: kept {len(table)} of {before} rows")

    print(f"Done: {len(cleaned_tables)} of {len(uploaded_files)} files processed.")
    if cleaned_tables:
        first_name = next(iter(cleaned_tables))
        print(f"Preview of the first file ({first_name}):")
        display(cleaned_tables[first_name].head())

## Step 4: Create the Excel Files

Each cleaned table is written to its own Excel file, named after the HTML file it came from (for example `WebDataRA-2022-09-21T11_52_44.xlsx`). If Excel files give you trouble, change `output_format` to `"csv"` to get CSV files instead, which Excel can open as well.

In [ ]:
output_format = "xlsx"   # "xlsx" or "csv"

output_files = {}        # new filename -> file content (bytes)

if not cleaned_tables:
    print("Nothing to convert yet. Complete Steps 2 and 3 first.")
else:
    for filename, table in cleaned_tables.items():
        base_name = filename.rsplit(".", 1)[0]
        if output_format == "csv":
            output_files[base_name + ".csv"] = table.to_csv(index=False).encode("utf-8")
        else:
            buffer = io.BytesIO()
            table.to_excel(buffer, index=False, engine="openpyxl")
            output_files[base_name + ".xlsx"] = buffer.getvalue()
    print(f"Created {len(output_files)} {output_format} file(s).")

## Step 5: Download Your Results

Because this notebook runs inside your browser, there is no folder to write to. Running the cell below creates download links:
- one link for **all files together** as a `.zip` archive (recommended if you uploaded several files)
- one link **per file** (click "Show individual files")

Click a link to save the file to your computer.

In [ ]:
def _make_link(parent, name, data_bytes, mime, label):
    blob = Blob.new([to_js(data_bytes)], to_js({"type": mime}, dict_converter=Object.fromEntries))
    link = document.createElement("a")
    link.href = URL.createObjectURL(blob)
    link.download = name
    link.textContent = label
    item = document.createElement("div")
    item.appendChild(link)
    parent.appendChild(item)

if not output_files:
    print("Nothing to download yet. Complete Steps 2 to 4 first.")
else:
    display(HTML('<div id="wdra-download-box"></div>'))
    box = document.getElementById("wdra-download-box")
    if box is None:
        box = document.body

    if len(output_files) > 1:
        zip_buffer = io.BytesIO()
        with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
            for name, data in output_files.items():
                zf.writestr(name, data)
        _make_link(box, "wdra_tweets.zip", zip_buffer.getvalue(), "application/zip",
                   f"Download all {len(output_files)} files as wdra_tweets.zip")

    details = document.createElement("details")
    summary = document.createElement("summary")
    summary.textContent = "Show individual files"
    details.appendChild(summary)
    mime_type = "text/csv" if output_format == "csv" else "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
    for name, data in output_files.items():
        _make_link(details, name, data, mime_type, f"Download {name}")
    box.appendChild(details)
    print("Click the links above to save your results.")